# Gestión de Permisos — Unity Catalog
Configuración de accesos para el usuario `yinfaulk_test_24@hotmail.com` sobre el catálogo `cat_ftr_smartdata_dev`.
Cubre todas las capas Medallion (Bronze, Silver, Gold), ejecución del job de orquestación y Delta Sharing para Power BI Desktop.

## 1. Usuario: yinfaulk_test_24@hotmail.com
Acceso completo: lectura y escritura en todas las capas, ejecución del job de orquestación y gestión de Delta Sharing.

In [0]:
-- Acceso al catálogo principal del proyecto
GRANT USE CATALOG ON CATALOG cat_ftr_smartdata_dev TO `yinfaulk_test_24@hotmail.com`;

In [0]:
-- Acceso a todos los esquemas del pipeline Medallion (raw, bronze, silver, golden)
GRANT USE SCHEMA ON SCHEMA cat_ftr_smartdata_dev.raw    TO `yinfaulk_test_24@hotmail.com`;
GRANT USE SCHEMA ON SCHEMA cat_ftr_smartdata_dev.bronze TO `yinfaulk_test_24@hotmail.com`;
GRANT USE SCHEMA ON SCHEMA cat_ftr_smartdata_dev.silver TO `yinfaulk_test_24@hotmail.com`;
GRANT USE SCHEMA ON SCHEMA cat_ftr_smartdata_dev.golden TO `yinfaulk_test_24@hotmail.com`;

In [0]:
-- Bronze: lectura, escritura y creación (necesario para las ejecuciones del pipeline)
GRANT SELECT, MODIFY ON TABLE cat_ftr_smartdata_dev.bronze.productos TO `yinfaulk_test_24@hotmail.com`;
GRANT SELECT, MODIFY ON TABLE cat_ftr_smartdata_dev.bronze.ventas    TO `yinfaulk_test_24@hotmail.com`;
GRANT CREATE TABLE   ON SCHEMA cat_ftr_smartdata_dev.bronze          TO `yinfaulk_test_24@hotmail.com`;

-- Silver: lectura, escritura y creación
GRANT SELECT, MODIFY ON TABLE cat_ftr_smartdata_dev.silver.ventas_productos_categorias TO `yinfaulk_test_24@hotmail.com`;
GRANT CREATE TABLE   ON SCHEMA cat_ftr_smartdata_dev.silver                            TO `yinfaulk_test_24@hotmail.com`;

-- Gold: lectura, escritura y creación (3 tablas, incluye kpi_vendedor_mes)
GRANT SELECT, MODIFY ON TABLE cat_ftr_smartdata_dev.golden.ventas_diarias_tienda   TO `yinfaulk_test_24@hotmail.com`;
GRANT SELECT, MODIFY ON TABLE cat_ftr_smartdata_dev.golden.ventas_categoria_mes    TO `yinfaulk_test_24@hotmail.com`;
GRANT SELECT, MODIFY ON TABLE cat_ftr_smartdata_dev.golden.kpi_vendedor_mes        TO `yinfaulk_test_24@hotmail.com`;
GRANT CREATE TABLE   ON SCHEMA cat_ftr_smartdata_dev.golden                        TO `yinfaulk_test_24@hotmail.com`;

## 2. Delta Sharing para Power BI Desktop
Permisos a nivel de metastore para que el usuario pueda crear shares (publicar tablas Gold) y recipients (generar tokens de acceso para Power BI Desktop).
El usuario ya tiene `SELECT` sobre las tablas Gold (concedido en la sección anterior), lo cual es necesario para poder añadirlas a un share.

In [0]:
-- Delta Sharing: permisos a nivel de metastore para gestionar el flujo completo desde Power BI
-- CREATE SHARE     : autoriza crear nuevos shares y añadir tablas Gold
-- CREATE RECIPIENT : autoriza generar el credential file / token para Power BI Desktop
GRANT CREATE SHARE     ON METASTORE TO `yinfaulk_test_24@hotmail.com`;
GRANT CREATE RECIPIENT ON METASTORE TO `yinfaulk_test_24@hotmail.com`;

## 3. Grupo Developers
Grupo para desarrolladores del proyecto. Incluye a `yinfaulk_test_24@hotmail.com` con acceso de lectura a todas las capas.

In [0]:
CREATE GROUP `Developers`;

In [0]:
ALTER GROUP `Developers`
ADD USER `yinfaulk_test_24@hotmail.com`;

In [0]:
GRANT USE CATALOG ON CATALOG cat_ftr_smartdata_dev TO `Developers`;

In [0]:
-- Acceso a esquemas para el grupo Developers (bronze, silver y golden)
GRANT USE SCHEMA ON SCHEMA cat_ftr_smartdata_dev.bronze TO `Developers`;
GRANT USE SCHEMA ON SCHEMA cat_ftr_smartdata_dev.silver TO `Developers`;
GRANT USE SCHEMA ON SCHEMA cat_ftr_smartdata_dev.golden TO `Developers`;
GRANT CREATE     ON SCHEMA cat_ftr_smartdata_dev.bronze TO `Developers`;

In [0]:
-- Lectura de todas las tablas del pipeline para el grupo Developers
GRANT SELECT ON TABLE cat_ftr_smartdata_dev.bronze.productos                   TO `Developers`;
GRANT SELECT ON TABLE cat_ftr_smartdata_dev.bronze.ventas                      TO `Developers`;
GRANT SELECT ON TABLE cat_ftr_smartdata_dev.silver.ventas_productos_categorias TO `Developers`;
GRANT SELECT ON TABLE cat_ftr_smartdata_dev.golden.ventas_diarias_tienda       TO `Developers`;
GRANT SELECT ON TABLE cat_ftr_smartdata_dev.golden.ventas_categoria_mes        TO `Developers`;
GRANT SELECT ON TABLE cat_ftr_smartdata_dev.golden.kpi_vendedor_mes            TO `Developers`;

## 4. Verificación de permisos
Consultas para confirmar los accesos configurados sobre catálogo, esquemas, tablas, external locations y Delta Sharing.

In [0]:
-- Verificar permisos sobre tablas Bronze (ejecutar después de correr el pipeline)
SHOW GRANTS ON TABLE cat_ftr_smartdata_dev.bronze.productos;
SHOW GRANTS ON TABLE cat_ftr_smartdata_dev.bronze.ventas;

In [0]:
-- Verificar permisos a nivel de esquema Bronze
SHOW GRANTS ON SCHEMA cat_ftr_smartdata_dev.bronze;

In [0]:
-- Verificar permisos a nivel de catálogo
SHOW GRANTS ON CATALOG cat_ftr_smartdata_dev;

In [0]:
-- Verificar permisos a nivel de esquemas Silver y Golden
SHOW GRANTS ON SCHEMA cat_ftr_smartdata_dev.silver;
SHOW GRANTS ON SCHEMA cat_ftr_smartdata_dev.golden;

In [0]:
-- Listar External Locations configuradas en el workspace
SHOW EXTERNAL LOCATIONS;

In [0]:
-- Verificar permisos sobre External Locations del pipeline
SHOW GRANTS ON EXTERNAL LOCATION `extl-raw`;

In [0]:
-- Listar usuarios del workspace
SHOW USERS;

In [0]:
-- Verificar permisos sobre las 3 tablas Gold (incluida la nueva kpi_vendedor_mes)
SHOW GRANTS ON TABLE cat_ftr_smartdata_dev.golden.ventas_diarias_tienda;
SHOW GRANTS ON TABLE cat_ftr_smartdata_dev.golden.ventas_categoria_mes;
SHOW GRANTS ON TABLE cat_ftr_smartdata_dev.golden.kpi_vendedor_mes;

In [0]:
-- Verificar los shares y recipients de Delta Sharing configurados en el metastore
SHOW SHARES;
SHOW RECIPIENTS;